# 09 ? Estimativas AIPW temporais do SINASC

**Estado cient?fico:** estimativas anuais independentes, sob as hip?teses de identifica??o. Plano congelado no commit `61efda5`; 2024 ? controle positivo obrigat?rio. Nenhum resultado deste notebook prova causalidade.

**Leitura r?pida:** o resumo e os gr?ficos abaixo s?o produzidos apenas ap?s o gate de 2024. A janela principal ? 2014?2024 (v1.1+H1); 2018?2024 (v1 estrito) ? sensibilidade obrigat?ria.

## 1. Objetivo e contrato

Contraste m?dio do risco de baixo peso ao nascer entre in?cio do pr?-natal at? o terceiro m?s (T=1) e in?cio posterior (T=0), na popula??o eleg?vel de nascidos vivos de cada ano. Estimativas em pontos percentuais. [Plano pr?-especificado](../../docs/temporal/PLANO_NOTEBOOK_09.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, Markdown
from src.estimativas_temporais import RAIZ, executar_analise
from src.visualizacao import aplicar_tema_ipt
raiz = RAIZ
pasta = raiz / 'outputs/temporal/estimativas'
assert (raiz / 'docs/temporal/PLANO_NOTEBOOK_09.md').is_file()
print('Raiz:', raiz)
print('Fase:', 'AIPW anual, sem pooled')

Raiz: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml
Fase: AIPW anual, sem pooled


## 2. Popula??es e janelas

Unidade: registro de nascido vivo. Chave temporal `(ano, UF_RESIDENCIA, CONTADOR)`. Em 2024 o AIPW hist?rico usa `ORDER BY contador`; em 2014?2023 a ordem ? `UF_RESIDENCIA, CAST(CONTADOR AS VARCHAR)`. As duas janelas s?o fixas e os anos comuns reutilizam exatamente a mesma linha anual.

## 3. Especifica??es C1, C2 e C3

| C?digo | Propensity | Outcomes | Papel |
|---|---|---|---|
| C1 | Log?stica L2 C=1 | Log?stica hist?rica por bra?o | principal |
| C2 | Mesma log?stica C=1 | HGB hist?rico por bra?o | alternativa principal de outcome |
| C3 | HGB hist?rico | Mesmos HGB de C2 | sensibilidade de propensity |

Sete X pr?-tratamento certificadas; nenhum tuning anual.

## 4. Cross-fitting e infer?ncia

Tr?s folds estratificados por `2*T+Y`, seed `20240925`. Cada nuisance ? ajustado apenas no treino; cada observa??o recebe uma previs?o OOF por modelo. O AIPW e seu SE iid reconciliam o hist?rico; o IC municipal usa `CODMUNRES` e quantil t com `G-1` graus de liberdade. ICs anuais n?o s?o um teste formal de diferen?a temporal.

## 5. Controle positivo 2024 e or?amento

A execu??o abaixo reusa caches hist?ricos somente se hashes, chaves, ordem, folds e par?metros conciliarem. Interrompe os anos anteriores se o controle positivo ou o or?amento de seis horas falhar. Checkpoints novos ficam em `data/interim/estimativas_temporais/`.

In [2]:
resultado = executar_analise(raiz)
controle = resultado['controle_2024']
print('Controle 2024:', controle['status'])
print('Execu??o:', resultado['execucao'])
print('Baseline:', resultado['baseline']['status'])
if 'orcamento' in resultado:
    print('Proje??o restante (h):', round(resultado['orcamento']['restante_2014_2023_horas'], 2))
assert controle['status'] == 'CONTROLE_POSITIVO_2024_APROVADO', controle['falhas']

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2024 | preparo | N=2251570 | 27.1s | RSS=1608232960


2024 | controle positivo | CONTROLE_POSITIVO_2024_APROVADO | 84.3s | RSS=1608232960


2024 | orçamento | restante=1.19h | RAM segura=True | disco seguro=True

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2014 | preparo | N=2691119 | 23.1s | RSS=2186424320


... (203 outputs left out)

2023 | m1_C2 | fold 3/3 | 22.2s | RSS=2423324672 | checkpoint OK


2023 | e_C3 | fold 3/3 | 24.7s | RSS=2423324672 | checkpoint OK


2023 | ano completo | SINALIZADO | 368.5s | RSS=2423324672 | cache=44619110


Controle 2024: CONTROLE_POSITIVO_2024_APROVADO
Execu??o: EXECUCAO_TEMPORAL_COMPLETA
Baseline: BASELINE_2024_PRESERVADO
Proje??o restante (h): 1.19


## 6. Diagn?sticos anuais dos nuisances

Log-loss, Brier, AUC, calibra??o, SMD, ESS e caudas s?o diagn?sticos preditivos/de suporte. N?o selecionam modelos pelo efeito.

In [3]:
diag = pd.read_csv(pasta / 'diagnosticos_nuisance_anual.csv')
execucoes = pd.read_csv(pasta / 'execucao_anos.csv')
display(execucoes[['ano','status','tempo_total_segundos','rss_pico_bytes']].sort_values('ano').reset_index(drop=True))
display(diag.loc[diag.alvo.eq('propensity'), ['ano','modelo','log_loss','brier','auc','erro_calibracao_medio','max_abs_smd_ponderado','ess_t0_sobre_n','ess_t1_sobre_n','fora_005_095_fracao']].sort_values(['ano','modelo']).reset_index(drop=True))

,ano,status,tempo_total_segundos,rss_pico_bytes
0,2014,SINALIZADO,429.522496,2186424320
1,2015,SINALIZADO,431.649195,2307731456
2,2016,SINALIZADO,405.565540,2307731456
3,2017,SINALIZADO,413.822954,2307731456
4,2018,SINALIZADO,420.585254,2321272832
5,2019,SINALIZADO,401.126088,2423324672
6,2020,SINALIZADO,374.744994,2423324672
7,2021,SINALIZADO,366.732140,2423324672
8,2022,SINALIZADO,378.146696,2423324672
9,2023,SINALIZADO,368.527696,2423324672


,ano,modelo,log_loss,brier,auc,erro_calibracao_medio,max_abs_smd_ponderado,ess_t0_sobre_n,ess_t1_sobre_n,fora_005_095_fracao
0,2014,hgb_historico,0.478496,0.154829,0.695168,0.000913,0.009949,0.684267,0.975055,0.047102
1,2014,logistica_C1,0.481811,0.155934,0.687869,0.000619,0.017823,0.692837,0.976409,0.038734
2,2015,hgb_historico,0.469569,0.150886,0.692744,0.001039,0.008506,0.696469,0.977090,0.039569
3,2015,logistica_C1,0.472612,0.151909,0.685675,0.001683,0.013871,0.694965,0.978718,0.039703
4,2016,hgb_historico,0.457518,0.146026,0.695883,0.000939,0.009269,0.681647,0.978099,0.056058
5,2016,logistica_C1,0.460831,0.147126,0.688124,0.001529,0.014789,0.683158,0.979850,0.051225
6,2017,hgb_historico,0.447236,0.141778,0.697855,0.001196,0.008005,0.679142,0.978701,0.057994
7,2017,logistica_C1,0.450222,0.142764,0.690770,0.001419,0.014835,0.675602,0.980336,0.056323
8,2018,hgb_historico,0.438054,0.137967,0.696933,0.001027,0.008031,0.682206,0.980244,0.060398
9,2018,logistica_C1,0.440836,0.138855,0.690121,0.001533,0.017676,0.676142,0.981876,0.060505


## 7. Estimativas C1 por ano

Principal sem trimming; o recorte `[0,05; 0,95]` altera a popula??o alvo e aparece separadamente.

In [4]:
aipw = pd.read_csv(pasta / 'aipw_anual.csv')
principal = aipw.loc[aipw.trimming.eq('sem_trimming') & aipw.especificacao.eq('C1')].sort_values('ano')
display(principal[['ano','n','n_t0','n_t1','estimativa_pp','se_cluster_pp','ic95_cluster_inferior_pp','ic95_cluster_superior_pp','status']].reset_index(drop=True))

,ano,n,n_t0,n_t1,estimativa_pp,se_cluster_pp,ic95_cluster_inferior_pp,ic95_cluster_superior_pp,status
0,2014,2691119,577127,2113992,-0.849446,0.053872,-0.955056,-0.743835,SINALIZADO
1,2015,2745846,565595,2180251,-0.881686,0.052184,-0.983986,-0.779386,SINALIZADO
2,2016,2626594,518310,2108284,-0.982854,0.072810,-1.125591,-0.840117,SINALIZADO
3,2017,2695676,511767,2183909,-1.003279,0.066539,-1.133722,-0.872836,SINALIZADO
4,2018,2734473,499451,2235022,-1.120190,0.073089,-1.263474,-0.976906,SINALIZADO
5,2019,2645383,465593,2179790,-1.103182,0.060673,-1.222125,-0.984240,SINALIZADO
6,2020,2535311,444287,2091024,-0.969413,0.068052,-1.102821,-0.836005,SINALIZADO
7,2021,2481267,399172,2082095,-1.108588,0.067727,-1.241360,-0.975816,SINALIZADO
8,2022,2397626,379207,2018419,-1.388228,0.069809,-1.525080,-1.251376,SINALIZADO
9,2023,2389346,353023,2036323,-1.322104,0.068176,-1.455756,-1.188453,SINALIZADO


## 8. Compara??o C1 e C2

C1 e C2 compartilham propensity log?stico; variam os outcomes hist?ricos.

In [5]:
comparacao = aipw.loc[aipw.trimming.eq('sem_trimming')].pivot(index='ano', columns='especificacao', values='estimativa_pp')
display(comparacao[['C1','C2']].round(3))

especificacao,C1,C2
ano,,
2014,-0.849,-0.778
2015,-0.882,-0.798
2016,-0.983,-0.891
2017,-1.003,-0.922
2018,-1.120,-1.042
2019,-1.103,-1.012
2020,-0.969,-0.903
2021,-1.109,-1.029
2022,-1.388,-1.297


## 9. Sensibilidade C3

C3 troca apenas o propensity para HGB hist?rico e reaproveita os outcomes C2. Falhas de C3 permanecem vis?veis como bloqueio da especifica??o.

In [6]:
display(aipw.loc[aipw.especificacao.eq('C3') & aipw.trimming.eq('sem_trimming'), ['ano','estimativa_pp','se_cluster_pp','status','motivo'] if 'motivo' in aipw else ['ano','estimativa_pp','se_cluster_pp','status']].sort_values('ano').reset_index(drop=True))

,ano,estimativa_pp,se_cluster_pp,status
0,2014,-0.768372,0.052465,SINALIZADO
1,2015,-0.799742,0.050626,SINALIZADO
2,2016,-0.887479,0.074156,SINALIZADO
3,2017,-0.911449,0.066736,SINALIZADO
4,2018,-1.035644,0.073113,SINALIZADO
5,2019,-1.003464,0.058706,SINALIZADO
6,2020,-0.900693,0.068447,SINALIZADO
7,2021,-1.025204,0.063674,SINALIZADO
8,2022,-1.288480,0.067895,SINALIZADO
9,2023,-1.224010,0.066864,SINALIZADO


## 10. Trimming

Estimativa principal sem recorte; sensibilidade fixa `[0,05; 0,95]`. O diagn?stico `[0,01; 0,99]` n?o determina um novo efeito principal.

In [7]:
display(aipw[['ano','especificacao','trimming','n','n_excluido','n_t0','n_t1','estimativa_pp','status']].sort_values(['ano','especificacao','trimming']).tail(18).reset_index(drop=True))

,ano,especificacao,trimming,n,n_excluido,n_t0,n_t1,estimativa_pp,status
0,2022,C1,0.05_0.95,2269444,128182,373348,1896096,-1.375782,SINALIZADO
1,2022,C1,sem_trimming,2397626,0,379207,2018419,-1.388228,SINALIZADO
2,2022,C2,0.05_0.95,2269444,128182,373348,1896096,-1.282099,SINALIZADO
3,2022,C2,sem_trimming,2397626,0,379207,2018419,-1.297357,SINALIZADO
4,2022,C3,0.05_0.95,2275594,122032,374350,1901244,-1.258581,SINALIZADO
5,2022,C3,sem_trimming,2397626,0,379207,2018419,-1.288480,SINALIZADO
6,2023,C1,0.05_0.95,2238015,151331,346086,1891929,-1.359922,SINALIZADO
7,2023,C1,sem_trimming,2389346,0,353023,2036323,-1.322104,SINALIZADO
8,2023,C2,0.05_0.95,2238015,151331,346086,1891929,-1.266106,SINALIZADO
9,2023,C2,sem_trimming,2389346,0,353023,2036323,-1.232121,SINALIZADO


## 11. Infer?ncia iid versus municipal

IC municipal ? a leitura prudente; o iid hist?rico aparece para reconcilia??o. Ambos s?o condicionais ?s hip?teses e n?o incorporam confundimento ausente.

In [8]:
display(aipw.loc[aipw.trimming.eq('sem_trimming'), ['ano','especificacao','estimativa_pp','se_iid_pp','se_cluster_pp','n_clusters']].sort_values(['ano','especificacao']).reset_index(drop=True))

,ano,especificacao,estimativa_pp,se_iid_pp,se_cluster_pp,n_clusters
0,2014,C1,-0.849446,0.045510,0.053872,5583
1,2014,C2,-0.778391,0.045489,0.053685,5583
2,2014,C3,-0.768372,0.045027,0.052465,5583
3,2015,C1,-0.881686,0.046084,0.052184,5587
4,2015,C2,-0.798415,0.046052,0.052521,5587
5,2015,C3,-0.799742,0.045252,0.050626,5587
6,2016,C1,-0.982854,0.048465,0.072810,5586
7,2016,C2,-0.890939,0.048451,0.074645,5586
8,2016,C3,-0.887479,0.047534,0.074156,5586
9,2017,C1,-1.003279,0.049373,0.066539,5581


## 12. S?rie temporal dos efeitos

Os gr?ficos usam Plotly com tema IPT; o eixo mostra pontos percentuais e os erros representam IC95% municipal. Linhas entre anos orientam a leitura, sem impor modelo de tend?ncia.

In [9]:
cores={'C1':'#00598E','C2':'#04B4E3','C3':'#9B3C8B'}
fig = go.Figure()
for spec in ('C1','C2','C3'):
    tab = aipw.loc[aipw.trimming.eq('sem_trimming') & aipw.especificacao.eq(spec) & aipw.estimativa_pp.notna()].sort_values('ano')
    fig.add_trace(go.Scatter(x=tab.ano, y=tab.estimativa_pp, mode='lines+markers', name=spec,
        line=dict(color=cores[spec]), marker=dict(size=8),
        error_y=dict(type='data', symmetric=False,
                     array=tab.ic95_cluster_superior_pp-tab.estimativa_pp,
                     arrayminus=tab.estimativa_pp-tab.ic95_cluster_inferior_pp),
        customdata=tab[['n','ess_t0','ess_t1']].to_numpy(),
        hovertemplate='Ano %{x}<br>Estimativa %{y:.3f} p.p.<br>N %{customdata[0]:,.0f}<br>ESS T0 %{customdata[1]:,.0f}<br>ESS T1 %{customdata[2]:,.0f}<extra>'+spec+'</extra>'))
fig.update_layout(xaxis_title='Ano', yaxis_title='Contraste AIPW (p.p.)', height=520)
fig.update_xaxes(dtick=1)
aplicar_tema_ipt(fig, 'Contraste AIPW anual e IC95% municipal')
fig.show()

In [10]:
fig2=go.Figure()
for spec in ('C1','C2'):
    tab=aipw.loc[aipw.trimming.eq('sem_trimming') & aipw.especificacao.eq(spec) & aipw.estimativa_pp.notna()].sort_values('ano')
    fig2.add_trace(go.Scatter(x=tab.ano,y=tab.estimativa_pp,mode='lines+markers',name=spec,line=dict(color=cores[spec]),
        hovertemplate='Ano %{x}<br>%{y:.3f} p.p.<extra>'+spec+'</extra>'))
fig2.update_layout(xaxis_title='Ano',yaxis_title='Contraste AIPW (p.p.)',height=430)
fig2.update_xaxes(dtick=1)
aplicar_tema_ipt(fig2,'C1 versus C2: mesmos escores, outcomes distintos')
fig2.show()

In [11]:
fig3=go.Figure()
for spec in ('C2','C3'):
    tab=aipw.loc[aipw.trimming.eq('sem_trimming') & aipw.especificacao.eq(spec) & aipw.estimativa_pp.notna()].sort_values('ano')
    fig3.add_trace(go.Scatter(x=tab.ano,y=tab.estimativa_pp,mode='lines+markers',name=spec,line=dict(color=cores[spec]),
        hovertemplate='Ano %{x}<br>%{y:.3f} p.p.<extra>'+spec+'</extra>'))
fig3.update_layout(xaxis_title='Ano',yaxis_title='Contraste AIPW (p.p.)',height=430)
fig3.update_xaxes(dtick=1)
aplicar_tema_ipt(fig3,'C2 versus C3: sensibilidade do propensity HGB')
fig3.show()

## 13. Janela 2018?2024

A sensibilidade v1 estrita ? um recorte das mesmas linhas anuais; n?o aciona novos fits.

In [12]:
j_principal=pd.read_csv(pasta/'aipw_janela_principal.csv')
j_sensibilidade=pd.read_csv(pasta/'aipw_sensibilidade_2018_2024.csv')
chaves=['ano','especificacao','trimming']
comparadas=j_sensibilidade.merge(j_principal,on=chaves,suffixes=('_s','_p'),validate='one_to_one')
assert len(comparadas)==len(j_sensibilidade)
for campo in ('estimativa_pp','se_iid_pp','se_cluster_pp','n'):
    assert comparadas[f'{campo}_s'].equals(comparadas[f'{campo}_p'])
print('Janela principal:',sorted(j_principal.ano.unique()))
print('Sensibilidade:',sorted(j_sensibilidade.ano.unique()),'| linhas id?nticas nos anos comuns')

Janela principal: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Sensibilidade: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)] | linhas id?nticas nos anos comuns


## 14. Leitura de 2021?2022

Comparar os contrastes e o suporte nesses anos ? s?rie completa. Uma eventual mudan?a ? heterogeneidade temporal aparente. N?o h? teste formal de quebra, nem atribui??o autom?tica a COVID-19, Zika ou pol?ticas.

In [13]:
display(aipw.loc[aipw.ano.isin([2020,2021,2022,2023]) & aipw.trimming.eq('sem_trimming'), ['ano','especificacao','estimativa_pp','ic95_cluster_inferior_pp','ic95_cluster_superior_pp','fora_005_095_fracao','ess_t0_sobre_n','status']].sort_values(['ano','especificacao']).reset_index(drop=True))

,ano,especificacao,estimativa_pp,ic95_cluster_inferior_pp,ic95_cluster_superior_pp,fora_005_095_fracao,ess_t0_sobre_n,status
0,2020,C1,-0.969413,-1.102821,-0.836005,0.054223,0.697343,SINALIZADO
1,2020,C2,-0.902534,-1.035877,-0.769192,0.054223,0.697343,SINALIZADO
2,2020,C3,-0.900693,-1.034875,-0.766510,0.050089,0.710375,SINALIZADO
3,2021,C1,-1.108588,-1.241360,-0.975816,0.059529,0.710358,SINALIZADO
4,2021,C2,-1.029339,-1.158307,-0.900371,0.059529,0.710358,SINALIZADO
5,2021,C3,-1.025204,-1.150030,-0.900378,0.054590,0.723906,SINALIZADO
6,2022,C1,-1.388228,-1.525080,-1.251376,0.053462,0.724276,SINALIZADO
7,2022,C2,-1.297357,-1.433914,-1.160800,0.053462,0.724276,SINALIZADO
8,2022,C3,-1.288480,-1.421582,-1.155379,0.050897,0.736411,SINALIZADO
9,2023,C1,-1.322104,-1.455756,-1.188453,0.063336,0.730444,SINALIZADO


## 15. Limita??es

Confundimento residual, sele??o de nascidos vivos, mensura??o, composi??o anual e altera??es de codifica??o podem afetar os contrastes. O IC municipal n?o cobre automaticamente depend?ncia entre anos nem toda a incerteza dos nuisances. Comparar ICs visualmente n?o testa igualdade de efeitos.

## 16. Conclus?es condicionais e integridade

In [14]:
manifesto=json.loads((pasta/'manifesto_notebook_09.json').read_text(encoding='utf-8'))
assert manifesto['baseline']['status']=='BASELINE_2024_PRESERVADO'
print('Controle positivo:',manifesto['controle_2024']['status'])
print('Execu??o:',manifesto['execucao'])
print('Anos:',[(r['ano'],r['status']) for r in manifesto['anos']])
print('Sob as hip?teses de identifica??o, os contrastes AIPW anuais acima descrevem diferen?as ajustadas nas popula??es eleg?veis de cada ano. N?o constituem prova causal.')

Controle positivo: CONTROLE_POSITIVO_2024_APROVADO
Execu??o: EXECUCAO_TEMPORAL_COMPLETA
Anos: [(2024, 'CONTROLE_POSITIVO_2024_APROVADO'), (2014, 'SINALIZADO'), (2015, 'SINALIZADO'), (2016, 'SINALIZADO'), (2017, 'SINALIZADO'), (2018, 'SINALIZADO'), (2019, 'SINALIZADO'), (2020, 'SINALIZADO'), (2021, 'SINALIZADO'), (2022, 'SINALIZADO'), (2023, 'SINALIZADO')]
Sob as hip?teses de identifica??o, os contrastes AIPW anuais acima descrevem diferen?as ajustadas nas popula??es eleg?veis de cada ano. N?o constituem prova causal.
